# Phase 4 — 2026 FIFA World Cup Simulation

In this phase we will:
1. Load the saved XGBoost model and artifacts from Phase 3
2. Explore team names in the dataset and map them to the 2026 WC draw
3. Build the group stage and simulate all matches
4. Run 10,000 tournament simulations to estimate each team's championship probability

## Step 1 — Explore Team Names in the Dataset

Before building the groups file, we need to check **exactly** how team names are stored in `results_cleaned.csv`.  
The official FIFA draw uses names like "South Korea", "Ivory Coast", "DR Congo" — but our dataset may use different spellings.  
We'll print every team name that appears in a WC 2026 group so we can map them correctly.

In [1]:
import pandas as pd

# Load the cleaned results dataset
df = pd.read_csv("../data/processed/results_cleaned.csv")

# All unique teams that appear as home OR away
all_teams = sorted(set(df["home_team"].unique()) | set(df["away_team"].unique()))

print(f"Total unique teams in dataset: {len(all_teams)}")
print()

# Official 2026 WC draw — all 48 qualified teams
wc2026_official = [
    # Group A
    "Mexico", "South Africa", "South Korea", "Czech Republic",
    # Group B
    "Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland",
    # Group C
    "Brazil", "Morocco", "Haiti", "Scotland",
    # Group D
    "United States", "Paraguay", "Australia", "Turkey",
    # Group E
    "Germany", "Curaçao", "Ivory Coast", "Ecuador",
    # Group F
    "Netherlands", "Japan", "Sweden", "Tunisia",
    # Group G
    "Belgium", "Egypt", "Iran", "New Zealand",
    # Group H
    "Spain", "Cape Verde", "Saudi Arabia", "Uruguay",
    # Group I
    "France", "Senegal", "Iraq", "Norway",
    # Group J
    "Argentina", "Algeria", "Austria", "Jordan",
    # Group K
    "Portugal", "DR Congo", "Uzbekistan", "Colombia",
    # Group L
    "England", "Croatia", "Ghana", "Panama",
]

print("Checking which official names ARE in the dataset:")
found     = [t for t in wc2026_official if t in all_teams]
not_found = [t for t in wc2026_official if t not in all_teams]

print(f"\nFound ({len(found)}):")
for t in found:
    print(f"   {t}")

print(f"\nNOT found — need mapping ({len(not_found)}):")
for t in not_found:
    print(f"   {t}")

Total unique teams in dataset: 194

Checking which official names ARE in the dataset:

Found (40):
   Mexico
   South Africa
   Canada
   Bosnia and Herzegovina
   Qatar
   Switzerland
   Brazil
   Morocco
   Haiti
   Scotland
   Paraguay
   Australia
   Turkey
   Germany
   Ecuador
   Netherlands
   Japan
   Sweden
   Tunisia
   Belgium
   Egypt
   New Zealand
   Spain
   Saudi Arabia
   Uruguay
   France
   Senegal
   Iraq
   Norway
   Argentina
   Algeria
   Austria
   Jordan
   Portugal
   Uzbekistan
   Colombia
   England
   Croatia
   Ghana
   Panama

NOT found — need mapping (8):
   South Korea
   Czech Republic
   United States
   Curaçao
   Ivory Coast
   Iran
   Cape Verde
   DR Congo


In [2]:
# For each missing team, search the dataset for the closest match
search_hints = {
    "South Korea": "korea",
    "Czech Republic": "czech",
    "United States": ["united", "usa", "u.s"],
    "Curaçao": ["curacao", "cura"],
    "Ivory Coast": ["ivory", "cote", "côte"],
    "Iran": "iran",
    "Cape Verde": ["cape", "verde"],
    "DR Congo": ["congo", "dr"],
}

print("Searching dataset for closest matches:\n")
for official, hints in search_hints.items():
    if isinstance(hints, str):
        hints = [hints]
    matches = [t for t in all_teams if any(h.lower() in t.lower() for h in hints)]
    print(f"  '{official}'  →  dataset candidates: {matches if matches else '(none — debut or new name)'}")

Searching dataset for closest matches:

  'South Korea'  →  dataset candidates: (none — debut or new name)
  'Czech Republic'  →  dataset candidates: ['Czechoslovakia']
  'United States'  →  dataset candidates: ['United Arab Emirates']
  'Curaçao'  →  dataset candidates: (none — debut or new name)
  'Ivory Coast'  →  dataset candidates: (none — debut or new name)
  'Iran'  →  dataset candidates: (none — debut or new name)
  'Cape Verde'  →  dataset candidates: (none — debut or new name)
  'DR Congo'  →  dataset candidates: ['Congo']


In [3]:
# Deeper search — try alternative known FIFA names for the still-missing teams
deeper_hints = {
    "South Korea": ["korea", "republic of korea"],
    "Czech Republic / Czechia": ["czech", "bohemia"],
    "United States / USA": ["states", "usa", "u.s.a"],
    "Curaçao": ["curacao", "curaçao", "netherlands antilles"],
    "Ivory Coast": ["ivory", "cote d", "côte d"],
    "Iran": ["ir iran", "iran", "persia"],
    "Cape Verde": ["cape verde", "verde", "islands"],
}

print("Deeper search:\n")
for label, hints in deeper_hints.items():
    matches = [t for t in all_teams if any(h.lower() in t.lower() for h in hints)]
    print(f"  {label}  →  {matches if matches else '(truly not in dataset)'}")

# Also check former_names.csv to see if any aliases are listed there
print("\n\n--- Checking former_names.csv ---")
fn = pd.read_csv("../data/raw/former_names.csv")
print(fn.to_string())

Deeper search:

  South Korea  →  (truly not in dataset)
  Czech Republic / Czechia  →  ['Czechoslovakia']
  United States / USA  →  (truly not in dataset)
  Curaçao  →  (truly not in dataset)
  Ivory Coast  →  (truly not in dataset)
  Iran  →  (truly not in dataset)
  Cape Verde  →  ['British Virgin Islands', 'Cayman Islands', 'Cook Islands', 'Faroe Islands', 'Solomon Islands', 'Turks and Caicos Islands']


--- Checking former_names.csv ---
                current                                former  start_date    end_date
0                 Benin                               Dahomey  1959-11-08  1975-11-30
1          Burkina Faso                           Upper Volta  1960-04-14  1984-08-04
2               Curaçao                  Netherlands Antilles  1957-03-03  2010-10-10
3        Czechoslovakia                               Bohemia  1903-04-05  1919-01-01
4        Czechoslovakia                   Bohemia and Moravia  1939-01-01  1945-05-01
5        Czechoslovakia  Representatio

In [4]:
# Print former_names.csv more manageably
fn = pd.read_csv("../data/raw/former_names.csv")
print("Columns:", fn.columns.tolist())
print("Shape:", fn.shape)
print()
print(fn.head(30).to_string(index=False))

Columns: ['current', 'former', 'start_date', 'end_date']
Shape: (36, 4)

            current                               former start_date   end_date
              Benin                              Dahomey 1959-11-08 1975-11-30
       Burkina Faso                          Upper Volta 1960-04-14 1984-08-04
            Curaçao                 Netherlands Antilles 1957-03-03 2010-10-10
     Czechoslovakia                              Bohemia 1903-04-05 1919-01-01
     Czechoslovakia                  Bohemia and Moravia 1939-01-01 1945-05-01
     Czechoslovakia Representation of Czechs and Slovaks 1993-03-24 1993-11-17
           DR Congo                        Belgian Congo 1948-05-25 1956-01-02
           DR Congo                   Congo-Léopoldville 1963-04-12 1964-07-19
           DR Congo                       Congo-Kinshasa 1965-01-09 1970-11-24
           DR Congo                                Zaïre 1971-01-10 1997-04-27
           Djibouti                    French Somaliland 1

In [5]:
# Look up specific missing teams in former_names
fn = pd.read_csv("../data/raw/former_names.csv")
search_terms = ["korea", "czech", "iran", "ivory", "cape verde", "curacao", "cura", "congo", "antilles"]
for term in search_terms:
    rows = fn[fn.apply(lambda r: r.astype(str).str.lower().str.contains(term).any(), axis=1)]
    if not rows.empty:
        print(f"--- '{term}' ---")
        print(rows.to_string(index=False))
        print()

--- 'czech' ---
       current                               former start_date   end_date
Czechoslovakia                              Bohemia 1903-04-05 1919-01-01
Czechoslovakia                  Bohemia and Moravia 1939-01-01 1945-05-01
Czechoslovakia Representation of Czechs and Slovaks 1993-03-24 1993-11-17

--- 'cura' ---
current               former start_date   end_date
Curaçao Netherlands Antilles 1957-03-03 2010-10-10

--- 'congo' ---
 current             former start_date   end_date
DR Congo      Belgian Congo 1948-05-25 1956-01-02
DR Congo Congo-Léopoldville 1963-04-12 1964-07-19
DR Congo     Congo-Kinshasa 1965-01-09 1970-11-24
DR Congo              Zaïre 1971-01-10 1997-04-27

--- 'antilles' ---
current               former start_date   end_date
Curaçao Netherlands Antilles 1957-03-03 2010-10-10



In [6]:
# Final check: exact dataset names for the remaining uncertain teams
for term in ["korea", "iran", "czech", "ivory", "cote", "côte", "cape", "curacao", "curaçao", "congo", "dr congo"]:
    matches = [t for t in all_teams if term.lower() in t.lower()]
    if matches:
        print(f"'{term}' → {matches}")

'czech' → ['Czechoslovakia']
'congo' → ['Congo']


In [7]:
# Check the FIFA ranking file — it often uses different names than results.csv
ranking = pd.read_csv("../data/raw/fifa_ranking-2024-06-20.csv")
ranking_teams = sorted(ranking["country_full"].unique())

print("FIFA Ranking file — searching for missing teams:")
for term in ["korea", "iran", "czech", "ivory", "cote", "cape verde", "curacao", "cura", "congo", "dr congo"]:
    matches = [t for t in ranking_teams if term.lower() in t.lower()]
    if matches:
        print(f"  '{term}' → {matches}")

FIFA Ranking file — searching for missing teams:
  'korea' → ['Korea DPR', 'Korea Republic']
  'iran' → ['IR Iran']
  'czech' → ['Czechia', 'Czechoslovakia']
  'curacao' → ['Curacao']
  'cura' → ['Curacao']
  'congo' → ['Congo', 'Congo DR']


In [8]:
# Verify exact names in results_cleaned.csv for remaining uncertain teams
# Iran and South Korea are major nations — let's see what names they use
for term in ["ir iran", "iran", "korea", "south korea", "curaçao", "curacao", 
             "côte", "ivory coast", "cape verde", "dr congo", "czech republic"]:
    matches = [t for t in all_teams if term.lower() == t.lower()]
    partial = [t for t in all_teams if term.lower() in t.lower()]
    if matches:
        print(f"EXACT '{term}' → ✅ {matches}")
    elif partial:
        print(f"PARTIAL '{term}' → {partial}")
    else:
        print(f"MISSING '{term}' → ❌ not in results_cleaned.csv")

MISSING 'ir iran' → ❌ not in results_cleaned.csv
MISSING 'iran' → ❌ not in results_cleaned.csv
MISSING 'korea' → ❌ not in results_cleaned.csv
MISSING 'south korea' → ❌ not in results_cleaned.csv
MISSING 'curaçao' → ❌ not in results_cleaned.csv
MISSING 'curacao' → ❌ not in results_cleaned.csv
MISSING 'côte' → ❌ not in results_cleaned.csv
MISSING 'ivory coast' → ❌ not in results_cleaned.csv
MISSING 'cape verde' → ❌ not in results_cleaned.csv
MISSING 'dr congo' → ❌ not in results_cleaned.csv
MISSING 'czech republic' → ❌ not in results_cleaned.csv


In [9]:
# Phase 1 used former_names.csv to RENAME teams — check what "Iran" and "Korea" 
# were renamed TO in results_cleaned.csv
fn = pd.read_csv("../data/raw/former_names.csv")

# Find any row mentioning Iran or Korea in the 'former' column
for term in ["iran", "korea", "czech", "ivory", "cape verde", "congo", "curacao"]:
    rows = fn[fn["former"].str.lower().str.contains(term, na=False)]
    if not rows.empty:
        print(f"--- former name contains '{term}' ---")
        print(rows[["current", "former"]].to_string(index=False))
        print()

# Also check what "current" these map to
print("\nAll 'current' values in former_names:")
print(sorted(fn["current"].unique()))

--- former name contains 'czech' ---
       current                               former
Czechoslovakia Representation of Czechs and Slovaks

--- former name contains 'congo' ---
 current             former
DR Congo      Belgian Congo
DR Congo Congo-Léopoldville
DR Congo     Congo-Kinshasa


All 'current' values in former_names:
['Benin', 'Burkina Faso', 'Curaçao', 'Czechoslovakia', 'DR Congo', 'Djibouti', 'Egypt', 'Eswatini', 'Ghana', 'Guinea-Bissau', 'Guyana', 'Indonesia', 'Israel', 'Malawi', 'Malaysia', 'Myanmar', 'North Macedonia', 'Northern Ireland', 'Republic of Ireland', 'Russia', 'Samoa', 'Serbia', 'Sri Lanka', 'Suriname', 'Tanzania', 'Vanuatu', 'Zambia', 'Zimbabwe']


In [10]:
# Check the RAW results.csv — maybe Iran/Korea appear there but got renamed in Phase 1
raw = pd.read_csv("../data/raw/results.csv")
raw_teams = sorted(set(raw["home_team"].unique()) | set(raw["away_team"].unique()))

for term in ["iran", "korea", "czech republic", "ivory", "cape verde", "curacao", "curaçao", "dr congo"]:
    matches = [t for t in raw_teams if term.lower() in t.lower()]
    print(f"RAW '{term}' → {matches if matches else '❌ not in raw either'}")

RAW 'iran' → ['Iran']
RAW 'korea' → ['North Korea', 'South Korea', 'United Koreans in Japan']
RAW 'czech republic' → ['Czech Republic']
RAW 'ivory' → ['Ivory Coast']
RAW 'cape verde' → ['Cape Verde']
RAW 'curacao' → ❌ not in raw either
RAW 'curaçao' → ['Curaçao']
RAW 'dr congo' → ['DR Congo']


In [11]:
# Verify United States in raw vs cleaned
print("RAW 'united':", [t for t in raw_teams if "united" in t.lower()])
print("CLEANED 'united':", [t for t in all_teams if "united" in t.lower()])

# Root cause: Phase 1 merged results with FIFA rankings using team name as key.
# Rankings use "IR Iran", "Korea Republic", "United States", etc.
# Results use "Iran", "South Korea", "United States" — mismatch drops those rows.
# Let's verify by checking Phase 1 output columns
print("\nresults_cleaned.csv columns:", df.columns.tolist())
print("Sample rows with a known team:", df[df["home_team"]=="Brazil"].head(2).to_string())

RAW 'united': ['United Arab Emirates', 'United Koreans in Japan', 'United States', 'United States Virgin Islands']
CLEANED 'united': ['United Arab Emirates']

results_cleaned.csv columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'home_ranking', 'country_abrv_x', 'home_points', 'previous_points_x', 'rank_change_x', 'confederation_x', 'away_ranking', 'country_abrv_y', 'away_points', 'previous_points_y', 'rank_change_y', 'confederation_y']
Sample rows with a known team:            date home_team away_team  home_score  away_score tournament              city        country  neutral  home_ranking country_abrv_x  home_points  previous_points_x  rank_change_x confederation_x  away_ranking country_abrv_y  away_points  previous_points_y  rank_change_y confederation_y
111  1993-03-17    Brazil    Poland           2           2   Friendly    Ribeirão Preto         Brazil    False           3.0            BRA         56.0           

### Why are 8 teams missing from `results_cleaned.csv`?

Phase 1 merged `results.csv` with `fifa_ranking-2024-06-20.csv` using **team name as the join key**.  
The two files use different FIFA naming conventions for some nations:

| Official WC 2026 name | In `results.csv` (raw) | In FIFA ranking file | In `results_cleaned.csv` |
|---|---|---|---|
| United States | United States | United States | ❌ dropped |
| Iran | Iran | IR Iran | ❌ dropped |
| South Korea | South Korea | Korea Republic | ❌ dropped |
| Czech Republic | Czech Republic | Czechia | ❌ dropped |
| Ivory Coast | Ivory Coast | Côte d'Ivoire | ❌ dropped |
| Cape Verde | Cape Verde | Cape Verde | ❌ dropped |
| Curaçao | Curaçao | Curacao | ❌ dropped |
| DR Congo | DR Congo | Congo DR | ❌ dropped |

**Consequence for Phase 4:** These 8 teams have no historical feature data.  
We will handle them with a **fallback**: use the global average of each feature across all teams.  
This is a reasonable approximation — better than crashing or excluding them.

**40 of 48 WC teams** have full feature data. That's 83% coverage — solid for a simulation!

In [16]:
# trying to fix the missing teams from the data set
# Step 1: Map ranking file names → results.csv names for the 8 mismatched teams
ranking_to_results = {
    "Korea Republic":  "South Korea",
    "IR Iran":         "Iran",
    "Czechia":         "Czech Republic",
    "Côte d'Ivoire":   "Ivory Coast",
    "Congo DR":        "DR Congo",
    "Curacao":         "Curaçao",
    "USA":             "United States",   # ← add this
    "Cabo Verde":      "Cape Verde",      # ← add this
}
# Note: "United States" and "Cape Verde" have the SAME name in both files
# but still got dropped — we'll investigate why in Step 2

print("Name mapping ready:", ranking_to_results)


Name mapping ready: {'Korea Republic': 'South Korea', 'IR Iran': 'Iran', 'Czechia': 'Czech Republic', "Côte d'Ivoire": 'Ivory Coast', 'Congo DR': 'DR Congo', 'Curacao': 'Curaçao', 'USA': 'United States', 'Cabo Verde': 'Cape Verde'}


In [17]:
import pandas as pd

# Reload the raw files (not the cleaned/merged one)
results_raw    = pd.read_csv("../data/raw/results.csv")
ranking_raw    = pd.read_csv("../data/raw/fifa_ranking-2024-06-20.csv")

# Apply the mapping: rename ranking team names to match results.csv
ranking_raw["country_full_mapped"] = ranking_raw["country_full"].replace(ranking_to_results)

# Verify — these should now appear in both files
missing_8 = ["South Korea", "Iran", "Czech Republic", "Ivory Coast",
             "DR Congo", "Curaçao", "United States", "Cape Verde"]

print("Results raw — do the 8 appear?")
for team in missing_8:
    count = (results_raw["home_team"] == team).sum() + (results_raw["away_team"] == team).sum()
    print(f"  {team}: {count} matches")

print("\nRanking mapped — do the 8 appear?")
for team in missing_8:
    count = (ranking_raw["country_full_mapped"] == team).sum()
    print(f"  {team}: {count} ranking entries")

Results raw — do the 8 appear?
  South Korea: 1003 matches
  Iran: 608 matches
  Czech Republic: 356 matches
  Ivory Coast: 633 matches
  DR Congo: 520 matches
  Curaçao: 381 matches
  United States: 786 matches
  Cape Verde: 231 matches

Ranking mapped — do the 8 appear?
  South Korea: 333 ranking entries
  Iran: 333 ranking entries
  Czech Republic: 326 ranking entries
  Ivory Coast: 333 ranking entries
  DR Congo: 266 ranking entries
  Curaçao: 136 ranking entries
  United States: 333 ranking entries
  Cape Verde: 333 ranking entries


In [15]:
# United States and Cape Verde have 0 ranking entries — their names in the ranking
# file must be different from results.csv. Let's find the actual names.

for term in ["united", "usa", "u.s", "states", "cape", "verde", "island"]:
    matches = [t for t in ranking_raw["country_full"].unique() if term.lower() in t.lower()]
    if matches:
        print(f"'{term}' → {matches}")

'united' → ['United Arab Emirates']
'usa' → ['USA']
'verde' → ['Cabo Verde']
'island' → ['Solomon Islands', 'Faroe Islands', 'Cayman Islands', 'Cook Islands', 'British Virgin Islands', 'US Virgin Islands', 'Turks and Caicos Islands']


In [18]:
# Step 3: Filter results to only the 8 missing teams
missing_8 = ["South Korea", "Iran", "Czech Republic", "Ivory Coast",
             "DR Congo", "Curaçao", "United States", "Cape Verde"]

mask = results_raw["home_team"].isin(missing_8) | results_raw["away_team"].isin(missing_8)
results_8 = results_raw[mask].copy()

# Check the columns in the ranking file so we know what's available
print("Ranking file columns:", ranking_raw.columns.tolist())
print("Ranking file sample row:")
print(ranking_raw.head(2).to_string())
print(f"\nResults rows for the 8 teams: {len(results_8)}")

Ranking file columns: ['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date', 'country_full_mapped']
Ranking file sample row:
    rank       country_full country_abrv  total_points  previous_points  rank_change confederation   rank_date country_full_mapped
0  140.0  Brunei Darussalam          BRU           2.0              0.0          140           AFC  1992-12-31   Brunei Darussalam
1   33.0           Portugal          POR          38.0              0.0           33          UEFA  1992-12-31            Portugal

Results rows for the 8 teams: 4434


In [19]:
# Step 4: Build a clean ranking lookup table
# Keep only the columns we need, using country_full_mapped as the team name
ranking_lookup = ranking_raw[[
    "rank_date", "country_full_mapped", "country_abrv",
    "rank", "total_points", "previous_points", "rank_change", "confederation"
]].copy()

# Convert date columns to datetime — required for merge_asof
results_8["date"]           = pd.to_datetime(results_8["date"])
ranking_lookup["rank_date"] = pd.to_datetime(ranking_lookup["rank_date"])

# Sort both by date — merge_asof requires sorted data
results_8      = results_8.sort_values("date").reset_index(drop=True)
ranking_lookup = ranking_lookup.sort_values("rank_date").reset_index(drop=True)

print("Ready to merge. Sample ranking_lookup:")
print(ranking_lookup[ranking_lookup["country_full_mapped"] == "South Korea"].head(3).to_string())

Ready to merge. Sample ranking_lookup:
     rank_date country_full_mapped country_abrv  rank  total_points  previous_points  rank_change confederation
87  1992-12-31         South Korea          KOR  49.0          28.0              0.0           49           AFC
199 1993-08-08         South Korea          KOR  36.0          40.0             28.0          -13           AFC
458 1993-09-23         South Korea          KOR  36.0          39.0             40.0            0           AFC


In [21]:
# Step 5: merge_asof — attach home team ranking to each match
# rename BOTH date and team columns in ranking_lookup first
ranking_for_merge = ranking_lookup.rename(columns={
    "rank_date":           "date",
    "country_full_mapped": "team"       # ← this was missing before
})

home_ranked = pd.merge_asof(
    results_8.rename(columns={"home_team": "team"})[
        ["date", "team", "away_team", "home_score", "away_score",
         "tournament", "city", "country", "neutral"]
    ],
    ranking_for_merge,
    on="date",
    by="team",
    direction="backward"
).rename(columns={
    "team":            "home_team",
    "rank":            "home_ranking",
    "total_points":    "home_points",
    "previous_points": "previous_points_x",
    "rank_change":     "rank_change_x",
    "confederation":   "confederation_x",
    "country_abrv":    "country_abrv_x",
})

print(f"Rows after home merge: {len(home_ranked)}")
print(f"NaN in home_ranking: {home_ranked['home_ranking'].isna().sum()}")
print(home_ranked.head(3).to_string())

Rows after home merge: 4434
NaN in home_ranking: 1685
        date      home_team      away_team  home_score  away_score tournament       city        country  neutral country_abrv_x  home_ranking  home_points  previous_points_x  rank_change_x confederation_x
0 1885-11-28  United States         Canada           0           1   Friendly     Newark  United States    False            NaN           NaN          NaN                NaN            NaN             NaN
1 1886-11-25  United States         Canada           3           2   Friendly     Newark  United States    False            NaN           NaN          NaN                NaN            NaN             NaN
2 1916-08-20         Sweden  United States           2           3   Friendly  Stockholm         Sweden    False            NaN           NaN          NaN                NaN            NaN             NaN


In [22]:
# Check: how many NaN rows are from before the ranking era (pre-1993)?
nan_mask = home_ranked["home_ranking"].isna()

print(f"Total NaN in home_ranking: {nan_mask.sum()}")
print(f"  - Before 1993: {(nan_mask & (home_ranked['date'] < '1993-01-01')).sum()}")
print(f"  - 1993 or later: {(nan_mask & (home_ranked['date'] >= '1993-01-01')).sum()}")
print()

# Filter to 1993+ only — this is what Phase 3 used too
home_ranked_filtered = home_ranked[home_ranked["date"] >= "1993-01-01"].copy()
print(f"Rows after filtering to 1993+: {len(home_ranked_filtered)}")
print(f"NaN in home_ranking after filter: {home_ranked_filtered['home_ranking'].isna().sum()}")


Total NaN in home_ranking: 1685
  - Before 1993: 1586
  - 1993 or later: 99

Rows after filtering to 1993+: 2848
NaN in home_ranking after filter: 99


In [23]:
# Investigate the 99 post-1993 NaN rows — which teams and years?
nan_post93 = home_ranked_filtered[home_ranked_filtered["home_ranking"].isna()]

print("Teams with NaN home_ranking after 1993:")
print(nan_post93.groupby("home_team")["date"].agg(["count", "min", "max"]).to_string())


Teams with NaN home_ranking after 1993:
                                  count        min        max
home_team                                                    
Bonaire                               1 2012-07-15 2012-07-15
Catalonia                             1 2013-12-30 2013-12-30
Curaçao                              30 1995-05-14 2010-10-31
DR Congo                             26 1993-01-10 1999-10-30
French Guiana                         2 2014-09-07 2014-11-15
Galicia                               1 2008-12-27 2008-12-27
Gambia                                6 1997-11-28 2023-11-20
Guadeloupe                            2 2014-10-12 2022-03-23
Kyrgyzstan                            4 1997-06-04 2024-11-19
Martinique                            2 2017-06-22 2023-09-10
North Korea                           9 2003-10-27 2024-11-14
Saint Lucia                           4 2010-10-17 2024-11-18
Saint Vincent and the Grenadines      3 2012-10-25 2016-09-02
Serbia                        

In [24]:
# Drop the 99 rows where home_ranking is NaN after 1993
# These are either: early years before the team entered the FIFA ranking system,
# or non-FIFA regional teams (Catalonia, French Guiana, etc.) that have no ranking
home_ranked_clean = home_ranked_filtered.dropna(subset=["home_ranking"]).copy()

print(f"Rows before drop: {len(home_ranked_filtered)}")
print(f"Rows after drop:  {len(home_ranked_clean)}")
print(f"Dropped: {len(home_ranked_filtered) - len(home_ranked_clean)} rows")
print(f"\nNaN in home_ranking now: {home_ranked_clean['home_ranking'].isna().sum()}")

Rows before drop: 2848
Rows after drop:  2749
Dropped: 99 rows

NaN in home_ranking now: 0


In [25]:
# Step 6: merge_asof — attach away team ranking to each match
away_ranked = pd.merge_asof(
    home_ranked_clean.rename(columns={"away_team": "team"}),
    ranking_for_merge.rename(columns={
        "rank":            "away_ranking",
        "total_points":    "away_points",
        "previous_points": "previous_points_y",
        "rank_change":     "rank_change_y",
        "confederation":   "confederation_y",
        "country_abrv":    "country_abrv_y",
    }),
    on="date",
    by="team",
    direction="backward"
).rename(columns={"team": "away_team"})

print(f"Rows after away merge: {len(away_ranked)}")
print(f"NaN in away_ranking: {away_ranked['away_ranking'].isna().sum()}")

Rows after away merge: 2749
NaN in away_ranking: 118


In [26]:
# Investigate the 118 NaN rows in away_ranking
nan_away = away_ranked[away_ranked["away_ranking"].isna()]

print("Teams with NaN away_ranking:")
print(nan_away.groupby("away_team")["date"].agg(["count", "min", "max"]).to_string())

Teams with NaN away_ranking:
                                  count        min        max
away_team                                                    
Armenia                               1 1994-05-15 1994-05-15
Bonaire                               2 2011-12-02 2015-02-01
Curaçao                              23 1995-05-07 2010-10-13
Czech Republic                        1 1994-02-23 1994-02-23
DR Congo                             30 1993-03-01 1999-12-01
Gambia                                8 2000-05-04 2025-03-24
Guadeloupe                            2 2011-06-14 2018-11-19
Guam                                  1 1996-08-05 1996-08-05
Kyrgyzstan                            3 1997-06-09 2024-09-05
Martinique                            4 2003-07-14 2021-07-15
Moldova                               2 1994-04-16 1994-04-20
North Korea                          15 1993-10-25 2025-06-10
Saint Kitts and Nevis                 2 2023-06-16 2023-06-28
Saint Lucia                           1 2

In [27]:
# Drop rows where away_ranking is NaN — same reason as home side
# (non-FIFA regional teams, post-cutoff dates, or pre-independence era)
away_ranked_clean = away_ranked.dropna(subset=["away_ranking"]).copy()

print(f"Rows before drop: {len(away_ranked)}")
print(f"Rows after drop:  {len(away_ranked_clean)}")
print(f"Dropped: {len(away_ranked) - len(away_ranked_clean)} rows")
print(f"\nNaN in away_ranking now: {away_ranked_clean['away_ranking'].isna().sum()}")

Rows before drop: 2749
Rows after drop:  2631
Dropped: 118 rows

NaN in away_ranking now: 0


In [28]:
# Step 7: Check column alignment before assembling final DataFrame
print("Target columns (results_cleaned.csv):")
print(df.columns.tolist())
print()
print("Our columns (away_ranked_clean):")
print(away_ranked_clean.columns.tolist())

Target columns (results_cleaned.csv):
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'home_ranking', 'country_abrv_x', 'home_points', 'previous_points_x', 'rank_change_x', 'confederation_x', 'away_ranking', 'country_abrv_y', 'away_points', 'previous_points_y', 'rank_change_y', 'confederation_y']

Our columns (away_ranked_clean):
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'country_abrv_x', 'home_ranking', 'home_points', 'previous_points_x', 'rank_change_x', 'confederation_x', 'country_abrv_y', 'away_ranking', 'away_points', 'previous_points_y', 'rank_change_y', 'confederation_y']


In [29]:
# Step 8: Reorder columns to match results_cleaned.csv, then append and save
target_cols = df.columns.tolist()

new_rows = away_ranked_clean[target_cols].copy()

print(f"Original results_cleaned.csv rows: {len(df)}")
print(f"New rows to append (8 teams): {len(new_rows)}")

# Concat and sort by date
df_updated = pd.concat([df, new_rows], ignore_index=True)
df_updated["date"] = pd.to_datetime(df_updated["date"])
df_updated = df_updated.sort_values("date").reset_index(drop=True)

print(f"Combined rows: {len(df_updated)}")
print(f"Any NaN in key columns: home_ranking={df_updated['home_ranking'].isna().sum()}, away_ranking={df_updated['away_ranking'].isna().sum()}")

# Save over the existing file
df_updated.to_csv("../data/processed/results_cleaned.csv", index=False)
print("\nSaved → data/processed/results_cleaned.csv")

Original results_cleaned.csv rows: 24184
New rows to append (8 teams): 2631
Combined rows: 26815
Any NaN in key columns: home_ranking=0, away_ranking=0

Saved → data/processed/results_cleaned.csv


In [30]:
# Step 9: Update wc2026_groups.csv — all 8 teams now have history
groups_df = pd.read_csv("../data/raw/wc2026_groups.csv")
groups_df["has_history"] = True   # all 48 teams now have data

groups_df.to_csv("../data/raw/wc2026_groups.csv", index=False)
print(groups_df.to_string(index=False))
print(f"\nAll {groups_df['has_history'].sum()} teams have history data")
print("Saved → data/raw/wc2026_groups.csv")

group                   team  has_history
    A                 Mexico         True
    A           South Africa         True
    A            South Korea         True
    A         Czech Republic         True
    B                 Canada         True
    B Bosnia and Herzegovina         True
    B                  Qatar         True
    B            Switzerland         True
    C                 Brazil         True
    C                Morocco         True
    C                  Haiti         True
    C               Scotland         True
    D          United States         True
    D               Paraguay         True
    D              Australia         True
    D                 Turkey         True
    E                Germany         True
    E                Curaçao         True
    E            Ivory Coast         True
    E                Ecuador         True
    F            Netherlands         True
    F                  Japan         True
    F                 Sweden      

---

## Summary — Data Recovery for 8 Missing WC Teams

Before building the simulation, we discovered that **8 of the 48 qualified teams had zero rows** in `results_cleaned.csv`. The cause: Phase 1 joined `results.csv` and `fifa_ranking-2024-06-20.csv` on team name, but the two files used different FIFA naming conventions (e.g., *"South Korea"* vs *"Korea Republic"*, *"Iran"* vs *"IR Iran"*).

**Fix applied:**
1. Built a name-mapping dictionary (`ranking_to_results`) to unify the 8 mismatched names
2. Filtered raw match results to only those 8 teams (`results_8` — 4,434 rows)
3. Used `pd.merge_asof` with `direction="backward"` to attach the most recent FIFA ranking to each match date — for both home and away sides
4. Dropped rows with no ranking data (pre-1993 era and non-FIFA regional opponents)
5. Appended **~2,600 recovered rows** to `results_cleaned.csv` and updated `wc2026_groups.csv`

**Result:** All 48 WC 2026 teams now have full feature history for the simulation.

---